# ML Pipelines Playbook

## Business Context

Customer churn refers to customers who discontinue a service or stop doing business with a company.

Predicting churn enables businesses to proactively retain customers through targeted interventions such as discounts, support outreach, and contract incentives.

Reducing churn can significantly improve customer retention, customer lifetime value, and revenue stability.

---

## Objective

Build reliable and reusable machine learning preprocessing workflows for customer churn prediction.

The focus of this notebook is not model performance, but designing preprocessing systems that can be consistently applied during training and inference.

---

## Business Question

How can we transform raw customer data into a format suitable for machine learning while ensuring consistency, maintainability, and reproducibility?

---

## Notebook Scope

This notebook focuses on:

- handling mixed-type datasets
- preprocessing numerical features
- preprocessing categorical features
- managing missing values
- encoding categorical variables
- scaling numerical variables
- building reusable preprocessing workflows
- understanding ColumnTransformer

Model training, cross-validation, hyperparameter tuning, and model selection will be covered in later sections of the ML Pipelines playbook.

---

## Workflow Engineering Mindset

Throughout this notebook, we will continuously ask:

> How can we ensure that the same preprocessing logic is applied every time the data is used?

The goal is not simply to clean data, but to build preprocessing workflows that are reusable, maintainable, and suitable for real-world machine learning systems.

## Imports

In [1]:
# Standard library
import random

# Third-party libraries
import numpy as np
import pandas as pd

# Local modules
from ml_playbook.config import DATA_DIR, RANDOM_STATE

# Scikit-learn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

## Configuration

In [2]:
# Constants
TARGET_COL = "Churn"

# Paths
DATA_PATH = DATA_DIR / "customer_churn.csv"

# Reproducibility
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

## Data Loading and Initial Inspection

### Dataset Overview

The dataset contains customer demographic, service usage, billing, and account information along with a binary churn indicator.

Each row represents a unique customer.

### Initial Dataset Inspection

In this section, we load the dataset and perform a preliminary inspection to understand its size, structure, and feature types before designing preprocessing workflows.

### Feature Summary

The dataset contains the following features:

- `gender` – customer gender
- `SeniorCitizen` – senior citizen indicator
- `tenure` – number of months with the company
- `MonthlyCharges` – monthly service charges
- `TotalCharges` – total amount billed to the customer
- `InternetService` – internet service type
- `Contract` – customer contract type
- `PaymentMethod` – payment method used
- `Churn` – customer churn indicator

In [3]:
df = pd.read_csv(DATA_PATH, header=0)
print(f"Shape: {df.shape}")

Shape: (1200, 9)


In [4]:
# Dataset summary
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   gender           1200 non-null   object 
 1   SeniorCitizen    1200 non-null   int64  
 2   tenure           1200 non-null   int64  
 3   MonthlyCharges   1200 non-null   float64
 4   TotalCharges     1086 non-null   float64
 5   InternetService  1103 non-null   object 
 6   Contract         1200 non-null   object 
 7   PaymentMethod    1120 non-null   object 
 8   Churn            1200 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 84.5+ KB


### Observations

- The dataset contains a mix of numerical and categorical features.
- Missing values are present in both numerical and categorical columns.
- Different feature types require different preprocessing strategies.
- The dataset is well suited for demonstrating ColumnTransformer and reusable preprocessing workflows.
- A preprocessing system must ensure that all feature transformations are applied consistently across future datasets.

### Dataset Structure and Summary Statistics

In [5]:
feature_columns = [
    col for col in df.columns
    if col != TARGET_COL
]

### Feature Groups

Identify numerical and categorical features so that appropriate preprocessing workflows can be applied to each feature type.

In [6]:
# Features treated as categorical despite numeric storage
categorical_overrides = ["SeniorCitizen"]

# Numerical features
numerical_features = df[feature_columns].select_dtypes(
    include=np.number
).columns.tolist()

numerical_features = [
    col
    for col in numerical_features
    if col not in categorical_overrides
]

# Categorical features
categorical_features = df[feature_columns].select_dtypes(
    exclude=np.number
).columns.tolist()

categorical_features.extend(
    categorical_overrides
)

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['tenure', 'MonthlyCharges', 'TotalCharges']

Categorical Features:
['gender', 'InternetService', 'Contract', 'PaymentMethod', 'SeniorCitizen']


### Numerical Features

Review numerical feature characteristics to understand potential preprocessing requirements such as scaling, missing value handling, and data quality issues.

In [7]:
df[numerical_features].describe()

,tenure,MonthlyCharges,TotalCharges
count,1200.000000,1200.000000,1086.000000
mean,35.768333,71.536025,2542.810138
std,20.800927,29.009536,1940.867966
min,1.000000,18.000000,-451.660000
25%,17.000000,51.317500,1008.590000
50%,35.000000,71.170000,2199.740000
75%,54.000000,90.860000,3714.115000
max,72.000000,150.000000,9820.930000


### Categorical Features

Review categorical feature characteristics to understand category cardinality and potential encoding requirements.

In [8]:
df[categorical_features].describe(include="all")

,gender,InternetService,Contract,PaymentMethod,SeniorCitizen
count,1200,1103,1200,1120,1200.000000
unique,2,3,3,4,NaN
top,Female,Fiber optic,Month-to-month,Mailed check,NaN
freq,604,493,626,288,NaN
mean,NaN,NaN,NaN,NaN,0.155833
std,NaN,NaN,NaN,NaN,0.362848
min,NaN,NaN,NaN,NaN,0.000000
25%,NaN,NaN,NaN,NaN,0.000000
50%,NaN,NaN,NaN,NaN,0.000000
75%,NaN,NaN,NaN,NaN,0.000000


### Observations

- The dataset contains both numerical and categorical features, requiring different preprocessing strategies.
- Numerical features exist on different scales and may require scaling before model training.
- Categorical features have relatively low cardinality and are suitable for standard encoding approaches.
- Missing values are present in both numerical and categorical columns.
- Data quality issues are visible (e.g., negative values in TotalCharges), reinforcing the need for systematic preprocessing.
- Feature groups have been defined and will be used to construct reusable preprocessing workflows.

## Target Variable Analysis

The target variable is:

- `Churn`
    - `1` → Customer churned
    - `0` → Customer retained

In [9]:
target_counts = df[TARGET_COL].value_counts()

target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage": (
        target_counts / len(df) * 100
    ).round(2)
})

target_summary

,Count,Percentage
Churn,,
0,656,54.67
1,544,45.33


### Observations

- Both churn classes are represented in the dataset.
- No severe class imbalance is observed.
- The target distribution will be revisited during model evaluation and workflow validation.

## Missing Value Analysis

In [10]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (
        df.isnull().mean()* 100
        ).round(2)
})

missing_summary = (
    missing_summary
    .loc[missing_summary["Missing_Count"] > 0]
    .sort_values(
        by="Missing_Count",
        ascending=False
    )
)

missing_summary

,Missing_Count,Missing_Percentage
TotalCharges,114,9.50
InternetService,97,8.08
PaymentMethod,80,6.67


### Observations

- Missing values are present in three features: `TotalCharges`, `InternetService`, and `PaymentMethod`.
- Missing values occur in both numerical and categorical feature groups.
- Missing value handling must be incorporated into the preprocessing workflow before model training.
- Different feature types may require different imputation strategies.
- Manual handling of missing values can become difficult to maintain as datasets and workflows grow.

## Duplicate Record Analysis

In [11]:
duplicate_count = df.duplicated().sum()
duplicate_pct = df.duplicated().mean() * 100

print(f"Number of rows: {len(df):,}")
print(f"Number of duplicates: {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_pct:.2f}%")

Number of rows: 1,200
Number of duplicates: 0
Duplicate percentage: 0.00%


### Observations

- No duplicate records were identified in the dataset.
- Duplicate-related data quality issues are not a concern for this analysis.
- Duplicate validation remains an important part of the initial dataset assessment process.

## Why Mixed-Type Preprocessing Matters

Real-world datasets rarely contain a single feature type.

In this dataset:
- numerical features contain continuous values such as tenure and billing information
- categorical features contain labels such as contract type and payment method

Different feature types often require different preprocessing strategies.

For example:
- numerical features may require missing value imputation and scaling
- categorical features may require missing value imputation and encoding

A reliable machine learning workflow should apply the correct preprocessing strategy to each feature type while maintaining consistency across training and inference.

---

## The Problem with Manual Preprocessing

As datasets grow, preprocessing often involves multiple steps such as:

1. imputing numerical features
2. scaling numerical features
3. imputing categorical features
4. encoding categorical features

Managing these transformations manually can make workflows difficult to maintain and prone to errors.

A more reliable approach is to organize preprocessing steps into reusable workflows that can be applied consistently across training and inference.

## Building a Numerical Preprocessing Workflow

Numerical features require both missing value handling and feature scaling.

These operations are applied sequentially using a Pipeline, allowing multiple preprocessing steps to be organized into a reusable workflow.

In [12]:
numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

### Observations

- Missing values will be imputed using the median.
- Numerical features will be standardized to a common scale.
- Multiple preprocessing steps have been organized into a reusable workflow.

## Building a Categorical Preprocessing Workflow

Categorical features require missing value handling followed by encoding.

These operations are grouped into a separate preprocessing workflow for categorical variables.

In [13]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

### Observations

- Missing categorical values will be imputed using the most frequent category.
- Categorical variables will be converted into numerical representations using one-hot encoding.
- Unknown categories encountered during inference will be handled safely.

## Combining Workflows with ColumnTransformer

Different feature groups often require different preprocessing strategies.

ColumnTransformer enables multiple preprocessing workflows to be applied to the appropriate columns while treating the entire dataset as a single preprocessing system.

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

### Observations

- Numerical and categorical preprocessing have been separated into independent workflows.
- ColumnTransformer combines these workflows into a single reusable preprocessing system.
- Feature-specific preprocessing logic no longer needs to be managed manually.
- Additional preprocessing workflows can be incorporated without changing the overall design.

## Workflow Architecture

```text
                Raw Dataset
                      │
        ┌─────────────┴─────────────┐
        │                           │
Numerical Features          Categorical Features
        │                           │
Numerical Pipeline          Categorical Pipeline
        │                           │
        └─────────────┬─────────────┘
                      │
              ColumnTransformer
                      │
              Processed Features
```

## Applying the Preprocessing Workflow

In [15]:
X = df.drop(columns=TARGET_COL, axis=1)

print(f"Shape X: {X.shape}")

Shape X: (1200, 8)


In [16]:
X_processed = preprocessor.fit_transform(X)

print(f"Shape X_processed: {X_processed.shape}")
print(f"Type X_processed: {type(X_processed)}")

Shape X_processed: (1200, 17)
Type X_processed: <class 'numpy.ndarray'>


## Inspecting the Transformed Features

The transformed output no longer preserves the original DataFrame structure.

Feature names generated by ColumnTransformer can be recovered using `get_feature_names_out()` to improve interpretability and debugging.

In [17]:
X_processed_df = pd.DataFrame(
    X_processed,
    columns=preprocessor.get_feature_names_out()
)

X_processed_df.head()

,num__tenure,num__MonthlyCharges,num__TotalCharges,cat__gender_Female,cat__gender_Male,cat__InternetService_DSL,cat__InternetService_Fiber optic,cat__InternetService_No,cat__Contract_Month-to-month,cat__Contract_One year,cat__Contract_Two year,cat__PaymentMethod_Bank transfer,cat__PaymentMethod_Credit card,cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check,cat__SeniorCitizen_0,cat__SeniorCitizen_1
0,0.780659,-0.754067,0.144504,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1,-0.998849,-1.626903,-1.080033,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,1.742556,-0.328513,0.906018,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,1.213513,-0.740618,0.438650,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
4,-0.710280,1.744774,-0.109231,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0


### Observations

- The entire dataset can now be transformed using a single preprocessing object.
- Numerical and categorical transformations are applied automatically.
- The transformed output can now serve as input to downstream machine learning models.
- Feature names produced by the preprocessing workflow can be inspected for debugging and interpretability.

## Why ColumnTransformer Matters in Production

ColumnTransformer provides a structured way to apply different preprocessing workflows to different feature groups.

Compared with manually managing transformations, it:

- improves maintainability
- reduces implementation errors
- keeps preprocessing logic organized
- ensures consistent transformations during training and inference

These advantages become increasingly important as machine learning systems grow in complexity.

## Key Takeaways

- Real-world datasets often contain mixed feature types.
- Different feature groups require different preprocessing strategies.
- Multiple preprocessing steps can be organized into reusable workflows using Pipeline.
- ColumnTransformer enables different preprocessing workflows to be applied to different feature groups.
- Preprocessing logic should be treated as part of the machine learning system rather than isolated code snippets.
- Organizing preprocessing workflows improves maintainability and consistency.
- The same preprocessing system can later be reused consistently during model training and inference.